# Two-call LangGraph extensions

Companion to `problem_first_ai/LangChain_LangGraph_Demo.ipynb`. It picks up where your
notebook ends (cell 50). Plan and reasoning: `wiki/notebooks/langgraph-two-call-extension.md`.

* **Paste into your notebook:** the *Helpers* cell, then any *Step* section, after cell 50.
* **Or run this file top to bottom.** `OFFLINE = True` uses scripted replies: no OpenRouter
  credits are spent and no spans reach Arize. Set it to `False` to use your real model
  (needs your `.env`).

## Helpers (paste these)

An offline stand-in for `model`, so every step can be tested without spending credits.
`FakeModel.bind_tools` is a no-op. That lets `with_structured_output` work offline: ChatOpenRouter's
default structured-output method is a forced tool call, and the fake replays it with `tool_reply(...)`.

In [ ]:
from contextlib import contextmanager, nullcontext

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class FakeModel(GenericFakeChatModel):
    """Scripted replies; bind_tools is a no-op so with_structured_output works offline."""
    def bind_tools(self, tools, **kwargs):
        return self


def tool_reply(name, **args):
    """A scripted structured reply: the model 'calling' the schema named `name`."""
    return AIMessage("", tool_calls=[{"name": name, "args": args, "id": "call_1"}])


@contextmanager
def fake_model(*replies):
    """Swap the global `model` for scripted replies; restore the real one afterwards."""
    global model
    msgs = [r if isinstance(r, AIMessage) else AIMessage(r) for r in replies]
    real, model = model, FakeModel(messages=iter(msgs))
    try:
        yield
    finally:
        model = real


def demo(*replies):
    """Scripted replies when OFFLINE, the real model otherwise (for demo runs, not tests)."""
    return fake_model(*replies) if OFFLINE else nullcontext()


class PromptRecorder(BaseCallbackHandler):
    """Records the last message of every chat-model call made during a graph run."""
    def __init__(self):
        self.prompts = []

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.prompts.append(messages[0][-1].content)

## Setup and your baseline (companion only, so skip these when pasting)

Sets up `model` the way your cell 5 does, then copies your cells 15, 7, 30, 32, 38, 40 and 42
verbatim so the steps below have the same objects you already built.

In [ ]:
OFFLINE = True   # False = real OpenRouter calls with your cell-5 settings

if OFFLINE:
    model = FakeModel(messages=iter([]))   # placeholder; every run below scripts its own replies
else:
    import os
    from dotenv import load_dotenv
    from langchain.chat_models import init_chat_model

    load_dotenv()
    missing = [n for n in ("OPENROUTER_API_KEY",) if not os.environ.get(n)]
    if missing:
        raise RuntimeError(f"Add to .env and restart the kernel: {missing}")
    model = init_chat_model(
        "openai/gpt-5-mini",
        model_provider="openrouter",
        max_tokens=4000,
        reasoning={"effort": "minimal"},
    )

In [ ]:
# --- your notebook, cell 15 (verbatim) ---
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class NumberState(TypedDict):
    num_values: int
    generated_values: list[int]
    total: int

In [ ]:
# --- your notebook, cell 7 (verbatim) ---
from langchain_core.prompts import ChatPromptTemplate

fortune_prompt = ChatPromptTemplate([
    (
        "system",
        "You are a wise and mystical AI fortune teller. Your predictions are "
        "funny, slightly exaggerated, but insightful. Keep it to 1-2 sentences",
    ),
    ("human", "Please entertain the user with their fortune request: {question}"),
])

question = "What is the next trillion dollar idea?"
prompt = fortune_prompt.invoke({"question": question})
print(prompt)

In [ ]:
# --- your notebook, cell 30 (verbatim) ---
class JokeState(TypedDict):
    question: str
    fortune: str
    joke: str

joke_prompt = ChatPromptTemplate([
    (
        "system",
        "You are a stand-up comedian who tells hilarious jokes in a casual, "
        "witty style using AI terminology. Keep it short",
    ),
    ("human", "Tell a joke about this fortune telling: {fortune}."),
])

def generate_fortune(state: JokeState) -> dict:
    prompt = fortune_prompt.invoke({"question": state["question"]})
    return {"fortune": model.invoke(prompt).text}

def generate_joke(state: JokeState) -> dict:
    prompt = joke_prompt.invoke({"fortune": state["fortune"]})
    return {"joke": model.invoke(prompt).text}

In [ ]:
# --- your notebook, cell 32 (verbatim) ---
builder = StateGraph(JokeState)
builder.add_node("generate_fortune", generate_fortune)
builder.add_node("generate_joke", generate_joke)

builder.add_edge(START, "generate_fortune")
builder.add_edge("generate_fortune", "generate_joke")
builder.add_edge("generate_joke", END)

graph = builder.compile()

In [ ]:
# --- your notebook, cell 38 (verbatim) ---
class SentimentState(TypedDict):
    sentence: str
    sentiment: str
    message: str

sentiment_prompt = ChatPromptTemplate([
    ("system", "Analyze sentiment. Reply with ONLY 'happy' or 'sad'."),
    ("human", "{sentence}"),
])

happy_prompt = ChatPromptTemplate([
    ("system", "Generate a cheerful, uplifting message in 1 sentence."),
    ("human", "The person said: {sentence}"),
])

sad_prompt = ChatPromptTemplate([
    ("system", "Generate a comforting message in 1 sentence that says it's alright."),
    ("human", "The person said: {sentence}"),
])

In [ ]:
# --- your notebook, cell 40 (verbatim) ---
def analyze_sentiment(state: SentimentState) -> dict:
    prompt = sentiment_prompt.invoke({"sentence": state["sentence"]})
    sentiment = model.invoke(prompt).text.strip().lower()
    return {"sentiment": sentiment}

def happy_response(state: SentimentState) -> dict:
    prompt = happy_prompt.invoke({"sentence": state["sentence"]})
    return {"message": model.invoke(prompt).text}

def sad_response(state: SentimentState) -> dict:
    prompt = sad_prompt.invoke({"sentence": state["sentence"]})
    return {"message": model.invoke(prompt).text}

In [ ]:
# --- your notebook, cell 42 (verbatim) ---
def route_sentiment(state: SentimentState) -> str:
    return state["sentiment"]

sentiment_builder = StateGraph(SentimentState)
sentiment_builder.add_node("analyze_sentiment", analyze_sentiment)
sentiment_builder.add_node("happy_response", happy_response)
sentiment_builder.add_node("sad_response", sad_response)

sentiment_builder.add_edge(START, "analyze_sentiment")
sentiment_builder.add_conditional_edges(
    "analyze_sentiment",
    route_sentiment,
    {"happy": "happy_response", "sad": "sad_response"},
)
sentiment_builder.add_edge("happy_response", END)
sentiment_builder.add_edge("sad_response", END)

sentiment_graph = sentiment_builder.compile()

## Step 1: an offline test harness

Checks your two existing graphs with no credits spent. A `PromptRecorder` callback shows that node 2
really receives node 1's output. Re-run this cell after every change. (In your own notebook, run it
*before* cell 28, or the fake runs will show up in your Arize project too.)

In [ ]:
# fortune -> joke: node 2 reads node 1's output from state
rec = PromptRecorder()
with fake_model("FORTUNE-XYZ", "JOKE-1"):
    out = graph.invoke({"question": "q?"}, {"callbacks": [rec]})
assert out == {"question": "q?", "fortune": "FORTUNE-XYZ", "joke": "JOKE-1"}
assert "FORTUNE-XYZ" in rec.prompts[1], rec.prompts

# sentiment routing, both branches, including the async path from cell 50
with fake_model("happy", "CHEER"):
    assert sentiment_graph.invoke({"sentence": "s"})["message"] == "CHEER"
with fake_model("sad", "COMFORT"):
    assert (await sentiment_graph.ainvoke({"sentence": "s"}))["message"] == "COMFORT"

print("baseline checks passed")

## Step 2: harden the router

In cell 40, `.strip().lower()` handles whitespace and case, and nothing else. Any reply other than exactly
`happy` or `sad` (`"Happy."`, `"**happy**"`, `"neutral"`, `""`) reaches cell 42's path map and raises a bare
`KeyError` from inside LangGraph. The fix below:

* reduces the reply to letters only, then **validates it inside the node**. A failure means nothing
  gets written to state, which matters once a checkpointer is added in Step 4;
* raises `ValueError` and re-asks the model (up to 3 tries). LangGraph's default `retry_on` does **not**
  retry `ValueError`/`KeyError`, hence the explicit policy. The second, default `RetryPolicy()` keeps the
  normal retries for transient errors such as a 429 or a dropped connection;
* still fails loudly (a clear `ValueError`) if the model keeps answering off-list. Backoff is 0.5 s,
  doubling, plus up to 1 s of jitter, so a persistent failure raises after about 1.5–3.5 s.

You must re-run the builder: redefining `analyze_sentiment` alone leaves the compiled graph on the old node.

In [ ]:
import re
from typing import Literal

from langgraph.types import RetryPolicy


class SentimentState(TypedDict):
    sentence: str
    sentiment: Literal["happy", "sad"]   # documents the contract; TypedDict values aren't checked at runtime
    message: str


def analyze_sentiment(state: SentimentState) -> dict:
    prompt = sentiment_prompt.invoke({"sentence": state["sentence"]})
    reply = model.invoke(prompt).text
    label = re.sub(r"[^a-z]", "", reply.lower())   # "Happy." / "**happy**" / " HAPPY\n" -> "happy"
    if label not in ("happy", "sad"):                 # "not happy" -> "nothappy" -> rejected, not misrouted
        raise ValueError(f"Classifier replied {reply!r}; expected 'happy' or 'sad'")
    return {"sentiment": label}


def route_sentiment(state: SentimentState) -> Literal["happy", "sad"]:
    return state["sentiment"]


sentiment_builder = StateGraph(SentimentState)
sentiment_builder.add_node(
    "analyze_sentiment",
    analyze_sentiment,
    retry_policy=[RetryPolicy(max_attempts=3, retry_on=ValueError), RetryPolicy()],
)
sentiment_builder.add_node("happy_response", happy_response)
sentiment_builder.add_node("sad_response", sad_response)
sentiment_builder.add_edge(START, "analyze_sentiment")
sentiment_builder.add_conditional_edges(
    "analyze_sentiment",
    route_sentiment,
    {"happy": "happy_response", "sad": "sad_response"},
)
sentiment_builder.add_edge("happy_response", END)
sentiment_builder.add_edge("sad_response", END)
sentiment_graph = sentiment_builder.compile()

In [ ]:
# regression checks for the router (a few seconds, mostly retry backoff)
with fake_model("Happy.", "CHEER"):
    assert sentiment_graph.invoke({"sentence": "s"})["sentiment"] == "happy"

rec = PromptRecorder()
with fake_model("neutral", "sad", "COMFORT"):        # off-list once, then a valid label
    out = sentiment_graph.invoke({"sentence": "s"}, {"callbacks": [rec]})
assert out["sentiment"] == "sad" and len(rec.prompts) == 3, rec.prompts

try:
    with fake_model("neutral", "positive", "not happy"):
        sentiment_graph.invoke({"sentence": "s"})
    raise AssertionError("expected ValueError")
except ValueError as exc:
    print("persistent off-list reply ->", exc)

print("router checks passed")

## Step 3: generate, then critique

This has the same shape as your fortune → joke graph, but node 2 now **judges** node 1 instead of riffing
on it. `generate_fortune` from cell 30 is reused unchanged. The critic grades the fortune against the brief
already in cell 7's system prompt and returns a validated rubric via `with_structured_output`.

Two details:

* Bind the critic **inside** the node. A critic built at cell level keeps the real model, so it spends
  credits even inside `fake_model(...)`.
* If the model answers in prose instead of calling the schema, `parsed` is `None` *and* `parsing_error`
  is `None`. The node records the raw reply instead of crashing.

In [ ]:
from pydantic import BaseModel, Field


class FortuneCritique(BaseModel):
    score: int = Field(ge=1, le=10, description="How well the fortune meets the brief, 1-10")
    funny: bool
    insightful: bool
    fix: str = Field(description="The single most useful change to make")
    verdict: Literal["pass", "revise"]


class CritiqueState(TypedDict, total=False):
    question: str
    fortune: str            # written by generate_fortune (cell 30, reused as-is)
    critique: dict          # written by critique_fortune
    critique_error: str     # set instead of crashing when the critic's reply can't be parsed


critique_prompt = ChatPromptTemplate([
    (
        "system",
        "You are a tough editor of fortune-teller replies. The brief: funny, slightly "
        "exaggerated, but insightful, 1-2 sentences. Score the fortune against the brief. "
        "Use verdict 'revise' when the score is below 7.",
    ),
    ("human", "Question: {question}\n\nFortune to review: {fortune}"),
])


def critique_fortune(state: CritiqueState) -> dict:
    critic = model.with_structured_output(FortuneCritique, include_raw=True)   # bound per call
    prompt = critique_prompt.invoke({"question": state["question"], "fortune": state["fortune"]})
    out = critic.invoke(prompt)
    if out["parsed"] is None:   # parsing_error is None when the model answered in prose
        reason = out["parsing_error"] or f"no structured reply: {out['raw'].text[:80]!r}"
        return {"critique_error": str(reason)[:200]}
    return {"critique": out["parsed"].model_dump(), "critique_error": ""}


critique_builder = StateGraph(CritiqueState)
critique_builder.add_node("generate_fortune", generate_fortune)   # same node as the joke graph
critique_builder.add_node("critique_fortune", critique_fortune)
critique_builder.add_edge(START, "generate_fortune")
critique_builder.add_edge("generate_fortune", "critique_fortune")
critique_builder.add_edge("critique_fortune", END)
critique_graph = critique_builder.compile()

In [ ]:
# offline checks: a structured critique parses; a prose reply is recorded, not raised
with fake_model("FORTUNE-1", tool_reply("FortuneCritique", score=8, funny=True, insightful=True,
                                         fix="none", verdict="pass")):
    out = critique_graph.invoke({"question": "q?"})
assert out["critique"]["verdict"] == "pass" and out["critique_error"] == ""

with fake_model("FORTUNE-1", "looks fine to me"):
    assert critique_graph.invoke({"question": "q?"})["critique_error"].startswith("no structured reply")

# one demo run (2 OpenRouter calls when OFFLINE = False)
with demo("A cloud of GPUs foretells your fortune.", tool_reply("FortuneCritique", score=5, funny=False,
          insightful=True, fix="Add a punchline.", verdict="revise")):
    out = critique_graph.invoke({"question": question})
print("fortune :", out["fortune"])
print("critique:", out.get("critique") or out.get("critique_error"))

## Step 4: trace handling (alongside Arize)

Your cell 24 streams full `"values"`. Two layers your notebook doesn't have yet:

* `stream_mode="updates"`, which yields `{node: what it returned}` as each node finishes;
* a **checkpointer** plus a `thread_id`, so every super-step is saved and `get_state_history` can replay a run
  afterwards (for example: exactly which fortune did `generate_joke` read?).

With Arize on (your cell 28), OpenInference copies the `thread_id` into every span as `session.id`, so
Arize's Sessions view lines up with the history below. Runs without a config (cells 36, 46, 48, 50) have
`session.id = None`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

traced_graph = builder.compile(checkpointer=InMemorySaver())     # cell 32's builder, now with memory
config = {"configurable": {"thread_id": "fortune-run-1"}}

with demo("FORTUNE-FOR-TRACE", "JOKE-FOR-TRACE"):
    for step, update in enumerate(traced_graph.stream({"question": question}, config, stream_mode="updates"), 1):
        for node, delta in update.items():
            print(f"[step {step}] {node} wrote {list(delta)}: {str(delta)[:60]}")

history = list(traced_graph.get_state_history(config))           # newest first
for snap in reversed(history):
    print(snap.metadata["step"], "next:", snap.next, "keys:", list(snap.values))

before_joke = next(s for s in history if s.next == ("generate_joke",))
print("generate_joke read fortune =", str(before_joke.values["fortune"])[:40])

## Step 5: a bounded reflection loop

This combines your two skills: sequential edges (cell 32) and a conditional edge (cell 42). The flow is
generate → critique → (revise → critique)* → end. It is bounded three ways:

* `MAX_REVISIONS = 2` caps a run at **6 model calls** (1 generate + 3 critiques + 2 revisions);
* the router also stops when the critique can't be read;
* `recursion_limit` is passed on **every** invoke. LangGraph's default limit is 10,007 steps, so it is
  no safety net for a broken budget.

`start_fortune` resets the per-run fields. Without the reset, a reused `thread_id` would carry `revisions`
and the previous `critique` over from the last run.

In [ ]:
MAX_REVISIONS = 2
LOOP_LIMITS = {"recursion_limit": 12}   # pass on every reflect_graph call


class ReflectState(CritiqueState, total=False):
    revisions: int


revise_prompt = ChatPromptTemplate([
    ("system", fortune_prompt.messages[0].prompt.template),        # same persona as cell 7
    ("human", "Question: {question}\nYour last fortune: {fortune}\n"
              "Editor's note: {fix}\nRewrite the fortune to address the note."),
])


def revise_fortune(state: ReflectState) -> dict:
    prompt = revise_prompt.invoke({"question": state["question"], "fortune": state["fortune"],
                                   "fix": state["critique"]["fix"]})
    return {"fortune": str(model.invoke(prompt).text), "revisions": state.get("revisions", 0) + 1}


def start_fortune(state: ReflectState) -> dict:
    # cell 30's node + a reset of the per-run fields (a checkpointer keeps them per thread_id)
    return {**generate_fortune(state), "revisions": 0, "critique": None, "critique_error": ""}


def after_critique(state: ReflectState) -> Literal["revise_fortune", "__end__"]:
    c = state.get("critique")
    if state.get("critique_error") or c is None:
        return END                                     # can't act on a review we couldn't read
    wants_revision = c["verdict"] == "revise" or c["score"] < 7
    if wants_revision and state.get("revisions", 0) < MAX_REVISIONS:
        return "revise_fortune"
    return END


reflect_builder = StateGraph(ReflectState)
reflect_builder.add_node("generate_fortune", start_fortune)
reflect_builder.add_node("critique_fortune", critique_fortune)
reflect_builder.add_node("revise_fortune", revise_fortune)
reflect_builder.add_edge(START, "generate_fortune")                     # sequential (cell 32 skill)
reflect_builder.add_edge("generate_fortune", "critique_fortune")
reflect_builder.add_conditional_edges("critique_fortune", after_critique)   # conditional (cell 42 skill)
reflect_builder.add_edge("revise_fortune", "critique_fortune")          # the loop back
reflect_graph = reflect_builder.compile(checkpointer=InMemorySaver())
print(reflect_graph.get_graph().draw_mermaid())                          # text: works offline

In [ ]:
def crit(score, verdict, fix="Sharper punchline."):
    return tool_reply("FortuneCritique", score=score, funny=score >= 7, insightful=True, fix=fix, verdict=verdict)


def loop_run(thread, *replies):
    rec = PromptRecorder()
    with fake_model(*replies):
        out = reflect_graph.invoke({"question": "q?"},
                                   {"configurable": {"thread_id": thread}, "callbacks": [rec], **LOOP_LIMITS})
    return out, len(rec.prompts)


out, calls = loop_run("t-pass-after-1", "F1", crit(5, "revise"), "F2", crit(8, "pass"))
assert out["fortune"] == "F2" and out["revisions"] == 1 and calls == 4

out, calls = loop_run("t-never-passes", "F1", crit(3, "revise"), "F2", crit(4, "revise"), "F3", crit(5, "revise"))
assert out["revisions"] == MAX_REVISIONS and calls == 6            # the budget holds

out, calls = loop_run("t-garbled", "F1", "meh")
assert out["critique_error"] and calls == 2                        # stops instead of looping

out, calls = loop_run("t-pass-after-1", "G1", crit(9, "pass"))        # same thread as run 1, fresh budget
assert out["revisions"] == 0 and out["critique"]["score"] == 9 and calls == 2

print("reflection loop checks passed")

## Step 6: make cell 50 actually concurrent

`await a` followed by `await b` runs the second graph only after the first finishes, so cell 50 takes as
long as cells 46 and 48 together. `asyncio.gather` overlaps them and keeps the `ainvoke` you're learning.
`return_exceptions=True` keeps one bad input from losing the other result. If the graph has a
checkpointer, give each input its own `thread_id`.

Run it with `OFFLINE = False` to see the timing difference; scripted replies have no latency.

In [ ]:
import asyncio
import time

inputs = [{"sentence": "I got the job! I'm so excited!"},
          {"sentence": "I failed my exam and feel terrible."}]

if OFFLINE:
    print("Step 6 shows a timing difference only with the real model; set OFFLINE = False.")
else:
    t0 = time.perf_counter()
    for x in inputs:
        await sentiment_graph.ainvoke(x)
    sequential = time.perf_counter() - t0

    t0 = time.perf_counter()
    results = await asyncio.gather(*(sentiment_graph.ainvoke(x) for x in inputs), return_exceptions=True)
    concurrent = time.perf_counter() - t0

    print(f"sequential {sequential:.1f}s vs concurrent {concurrent:.1f}s")
    for x, r in zip(inputs, results):
        print(x["sentence"][:32], "->", r if isinstance(r, Exception) else (r["sentiment"], r["message"]))

## Step 7: point the same cells at a local model (Sovereign AI A/B)

Change only how `model` is created. Every node, test and graph above stays the same:

```python
model = init_chat_model("<served-model-name>", model_provider="openai",
                        base_url="http://localhost:8000/v1", api_key="EMPTY", max_tokens=4000)
```

Structured output takes a **different path** here. `ChatOpenAI` defaults to `method="json_schema"`
(`response_format`), while ChatOpenRouter uses a forced tool call. So either confirm your server supports
JSON-schema structured outputs (vLLM does), or bind with
`model.with_structured_output(FortuneCritique, method="function_calling", include_raw=True)` and enable
tool calling on the server. The offline fake only exercises the tool-call path, so make **one real
structured call** against the local server before trusting the comparison.